In [ ]:
import requests
import pandas as pd
from datetime import datetime
import time

In [ ]:
API_KEY = "62932d0d49cf4a2e9ce175747262804"
BASE_URL = "http://api.weatherapi.com/v1/"

In [ ]:
def get_city_data(city, api_key):
    today = datetime.now().strftime("%Y-%m-%d")

    try:
        # --- Combined Weather Call ---
        forecast_url = f"{BASE_URL}forecast.json?key={api_key}&q={city}&days=3&aqi=yes"
        forecast_res = requests.get(forecast_url)
        forecast_res.raise_for_status()

        # --- Astronomy Call ---
        astro_url = f"{BASE_URL}astronomy.json?key={api_key}&q={city}&dt={today}"
        astro_res = requests.get(astro_url)
        astro_res.raise_for_status()

        # Parse JSON
        data = forecast_res.json()
        astro_data = astro_res.json()

        location = data['location']
        current = data['current']
        forecast_days = data['forecast']['forecastday']
        astro = astro_data['astronomy']['astro']

        # --- Build Summary ---
        city_summary = {
            "timestamp": datetime.now().isoformat(),
            "city": location['name'],
            "region": location['region'],
            "country": location['country'],
            "temp_f": current['temp_f'],
            "feels_like": current['feelslike_f'],
            "condition": current['condition']['text'],
            "humidity": current['humidity'],
            "wind_mph": current['wind_mph'],
            "uv": current['uv'],
            "precip_in": current['precip_in'],
            "sunrise": astro['sunrise'],
            "sunset": astro['sunset'],
            "moon_phase": astro['moon_phase']
        }
        for i, day in enumerate(forecast_days):
            city_summary[f"date_{i+1}"] = day['date']
            city_summary[f"max_temp_day_{i+1}"] = day['day']['maxtemp_f']
            city_summary[f"min_temp_day_{i+1}"] = day['day']['mintemp_f']
            city_summary[f"avg_temp_day_{i+1}"] = day['day']['avgtemp_f']
        return city_summary

    except requests.exceptions.RequestException as e:
        print(f"Error fetching {city}: {e}")
        return None

In [ ]:
cities = [
    "Louisville",
    "Lexington",
    "Cincinnati",
    "Paducah",
    "Bowling Green"
]

In [ ]:
def build_dataset(cities, api_key):
    results = []

    for city in cities:
        print(f"Fetching data for {city}...")
        data = get_city_data(city, api_key)

        if data:
            results.append(data)

        # Prevent rate limiting
        time.sleep(1)

    df = pd.DataFrame(results)
    return df

In [ ]:
df = build_dataset(cities, API_KEY)
df

Fetching data for Louisville...
Fetching data for Lexington...
Fetching data for Cincinnati...
Fetching data for Paducah...
Fetching data for Bowling Green...


,timestamp,city,region,country,temp_f,feels_like,condition,humidity,wind_mph,uv,...,min_temp_day_1,avg_temp_day_1,date_2,max_temp_day_2,min_temp_day_2,avg_temp_day_2,date_3,max_temp_day_3,min_temp_day_3,avg_temp_day_3
0,2026-04-28T17:22:08.226657,Louisville,Kentucky,United States of America,75.0,78.1,Partly cloudy,60,8.5,3.5,...,61.3,67.9,2026-04-29,65.8,52.3,60.7,2026-04-30,63.7,43.3,54.0
1,2026-04-28T17:22:09.573466,Lexington-Fayette,Kentucky,United States of America,75.0,77.4,Sunny,62,12.1,1.5,...,59.4,67.9,2026-04-29,65.5,49.0,59.7,2026-04-30,60.0,39.3,49.8
2,2026-04-28T17:22:10.878318,Cincinnati,Ohio,United States of America,75.9,78.7,Partly cloudy,54,10.5,3.2,...,58.3,64.0,2026-04-29,60.5,47.7,56.3,2026-04-30,59.5,40.8,50.2
3,2026-04-28T17:22:13.275343,Paducah,Kentucky,United States of America,79.0,80.7,Partly cloudy,70,2.2,2.1,...,65.3,72.2,2026-04-29,67.3,50.7,61.7,2026-04-30,64.4,44.2,55.0
4,2026-04-28T17:22:14.626573,Bowling Green,Kentucky,United States of America,79.0,81.1,Partly cloudy,62,8.3,1.9,...,64.0,71.9,2026-04-29,66.3,51.2,61.5,2026-04-30,63.5,42.7,53.4


In [ ]:
df.to_csv("../data/API_weather_data.csv", index=False)